## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading data

In [ ]:
df = pd.read_csv('../../data/raw/billings.csv')
df.head()

## Initial Analysis

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Converting column names to snakecase

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

In [ ]:
from src import utils
df = utils.convert_columns_to_snake_case(df)

## Distribution of target variable

In [ ]:
plt.figure(figsize=(5, 4))
df['prospect_outcome'].value_counts().plot(kind='bar', color=['steelblue','tomato','gray'])
plt.title('Prospect Outcome Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Checking duplicate record

In [ ]:
df.duplicated(subset=['co_ref', 'renewal_year']).sum()

## Checking data types

In [ ]:
df.dtypes

## Date columns are converted from string to datetime

In [ ]:
df['registration_date'] = pd.to_datetime(df['registration_date'], dayfirst=True, errors='coerce')
df['proforma_date'] = pd.to_datetime(df['proforma_date'], dayfirst=True, errors='coerce')
df['prospect_renewal_date'] = pd.to_datetime(df['prospect_renewal_date'], dayfirst=True, errors='coerce')
df['last_renewal'] = pd.to_datetime(df['last_renewal'], dayfirst=True, errors='coerce')
df['renewal_month'] = pd.to_datetime(df['renewal_month'], dayfirst=True, errors='coerce')

## Checking null values

In [ ]:
(df.isnull().mean() * 100).sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(12, 4))
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
sns.barplot(x=null_pct.index, y=null_pct.values)
plt.xticks(rotation=45, ha='right')
plt.title('Missing Values by Column (%)')
plt.xlabel('Columns')
plt.ylabel('Null %')
plt.tight_layout()
plt.show()

## Dropping fully null column

In [ ]:
df.drop(columns=['last_years_date_paid'], inplace=True)

## Dropping >90% Null Columns

In [ ]:
df.drop(columns=['connection_net','connection_qty','starting_connection_net','starting_connection_qty'], inplace=True)

## Removal of Leakage Columns

Columns `Closed_Date`, `Prospect_Status`, `Total_Net_Paid`, and `Gross` are removed as they contain post-outcome information.

In [ ]:
df.drop(columns=[
    'closed_date',
    'prospect_status',
    'total_net_paid',
    'gross'
], inplace=True)

## Removal of free text column

`current_anchor_list` should be dropped because it is a free text column. Machine learning models cannot read text like this directly.

In [ ]:
df.drop(columns=['current_anchor_list'], inplace=True)

## Detecting duplicate columns

In [ ]:
duplicates = []
cols = df.columns.tolist()

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        c1, c2 = cols[i], cols[j]
        if df[c1].astype(str).equals(df[c2].astype(str)):
            duplicates.append((c1, c2))

for c1, c2 in duplicates:
    print(f"{c1}  ==  {c2}")

## Fixing of duplicate columns

In [ ]:
df.drop(columns=[
    'datetime_out',     # duplicate of Renewal_Month
    'of_connection',     # duplicate of Proforma_Approved_Lists
    'amount',           # duplicate of Starting_Net
    'connection_group',    # duplicate of Anchor_Group
], inplace=True)

## Removal of invalid pricing records

Records with `starting_net` less than or equal to zero are removed as they represent invalid or non-revenue cases.


In [ ]:
df = df[df['starting_net'] > 0]

## Handling missing values

In [ ]:
# Create a flag to identify first year customers
# If last_renewal is null then first year(1), else returning (0)
df['is_first_year'] = df['last_renewal'].isnull().astype(int)

# Fill missing previous payment with current price
# Assumes first year customers paid similar to current starting_net
df['last_total_net_paid'] =df['last_total_net_paid'].fillna(df['starting_net'])

# Fill missing previous connections with 0
# First year customers had no prior connections
df['last_connections'] = df['last_connections'].fillna(0)

# Fill missing previous band with current band
# Assumes no change in band for first year customers
df['last_band']=df['last_band'].fillna(df['band'])

# Fill missing previous renewal date with current renewal date
# Avoids issues in time based calculations for first year customers
df['last_renewal'] = df['last_renewal'].fillna(df['prospect_renewal_date'])

In [ ]:
# Null means no discount was given
df['has_discount'] = df['discount_amount'].notna().astype(int)

# Parse the % string to float for the rows that have it
df['discount_pct'] = (
    df['discount_amount']
    .str.replace('%', '', regex=False)
    .astype(float) / 100
)
df['discount_pct'] = df['discount_pct'].fillna(0)

df.drop(columns=['discount_amount'], inplace=True)

In [ ]:
df['payment_timeframe']= df['payment_timeframe'].fillna(0) # 0 = not yet paid

In [ ]:
# Fill missing values using current status

df['proforma_auto_renewal'] = df['proforma_auto_renewal'].fillna(
    df['current_auto_renewal_flag'].map({'y': True, 'n': False})
)
df['proforma_world_pay_token'] = df['proforma_world_pay_token'].fillna(
    df['current_world_pay_token'].map({'y': True, 'n': False})
)

In [ ]:
# Fill missing account stage and audit status with 'Unknown'

df['proforma_account_stage'] = df['proforma_account_stage'].fillna('Unknown')
df['proforma_audit_status'] = df['proforma_audit_status'].fillna('Unknown')

In [ ]:
# Replace 0 with NaN first 
df['last_years_price'] = df['last_years_price'].replace(0, np.nan) 

# Fill nulls with starting_net (assume no price change) 
df['last_years_price'] = df['last_years_price'].fillna(df['starting_net'])

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df['tenure_years'] = df['tenure_years'].fillna(df['tenure_years'].median())
df['tenure_group'] = df['tenure_group'].fillna(df['tenure_group'].mode()[0])
df['registration_date']=df['registration_date'].fillna(df['registration_date'].median())
df['proforma_date'] = df['proforma_date'].fillna(df['proforma_date'].median())
df['band'] = df['band'].fillna(df['band'].mode()[0])
df['last_band'] = df['last_band'].fillna(df['last_band'].mode()[0])

## Verifying after null handling

In [ ]:
(df.isnull().sum()).sort_values(ascending=False)

## Numerical columns are converted to integer type

In [ ]:
df['tenure_years'] = df['tenure_years'].astype(int)
df['last_connections'] = df['last_connections'].astype(int)
df['proforma_approved_lists'] = df['proforma_approved_lists'].astype(int)
df['payment_timeframe'] = df['payment_timeframe'].astype(int)

## Categorical features are converted category data type

In [ ]:
cat_cols = ['band', 'last_band', 'tenure_group', 'anchor_group', 'payment_method', 'proforma_account_stage', 'proforma_audit_status', 'proforma_membership_status'] 
for col in cat_cols: 
    df[col] = df[col].astype('category')

## Standardization of categorical data

In [ ]:
df['proforma_account_stage'] = df['proforma_account_stage'].str.strip().str.title()
(df['proforma_account_stage'].unique())

In [ ]:
df['payment_method'] = df['payment_method'].str.strip().str.title()
(df['payment_method'].unique())


In [ ]:
df['band'] = df['band'].str.strip().str.title()
df['last_band'] = df['last_band'].str.strip().str.title()
(df['band'].unique())

In [ ]:
df['tenure_group'] = df['tenure_group'].str.strip()
df['tenure_group'].unique()

In [ ]:
df['anchor_group'] = df['anchor_group'].str.strip()
df['anchor_group'].unique()

In [ ]:
df['proforma_membership_status'] = df['proforma_membership_status'].str.strip().str.title()
(df['proforma_membership_status'].unique())

## Final data shape

In [ ]:
df.shape

## Exporting cleaned dataset

In [ ]:
df.to_csv('../../data/interim/billings_cleaned.csv', index=False)

## Correlation Heatmap — Numeric Features

In [ ]:
num_df = df.select_dtypes(include='number').drop(
    columns=['renewal_year', 'is_first_year', 'has_discount'], errors='ignore')

plt.figure(figsize=(12, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

## Number of Renewals per Year

In [ ]:
plt.figure(figsize=(8, 4))
df['renewal_year'].value_counts().sort_index().plot(
    kind='bar')
plt.title('Number of Renewals per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Starting Net Price Distribution by Band

In [ ]:
plt.figure(figsize=(12, 5))
band_order = ['Band A','Band B','Band C1','Band C2','Band D',
              'Band E','Band F','Band F1','Band F2','Band G']
sns.boxplot(data=df[df['band'].isin(band_order)],
            x='band', y='starting_net', order=band_order, color='steelblue')
plt.title('Starting Net Price Distribution by Band')
plt.xlabel('Band')
plt.ylabel('Starting Net (£)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Tenure Years & Tenure Group Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Exact years
df['tenure_years'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Tenure Years Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Group
df['tenure_group'].value_counts().plot(
    kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Tenure Group Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()